# CivicPulse — Notebook 02: Preprocessing, Features & Embeddings
**Course Outcome Coverage:** CO2 (Configurable Preprocessing, Feature Extraction: BoW, TF-IDF, Word2Vec, GloVe, FastText, Contextual BERT Embeddings, Dimensionality Reduction: PCA, t-SNE, UMAP)
**Author:** CivicPulse Capstone Team
**Date:** October 2026

---

## 1. The Preprocessing Pipeline Architecture

Raw municipal reports and news feeds suffer from severe noise: HTML tags, raw URLs, contractions, mixed capitalization, non-standard abbreviations, and typos.
In `server/nlp/preprocess.py`, we built a modular pipeline where every transformation can be enabled, disabled, or audited.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from server.nlp.preprocess import TextPreprocessor, default_preprocessor
from server.nlp.features import (
    build_bow_vectorizer, build_tfidf_vectorizer,
    train_word2vec, get_word2vec_document_vector,
    train_fasttext, get_fasttext_document_vector,
    load_glove_vectors, get_glove_document_vector,
    get_bert_embeddings
)

print("Modules initialized.")

## 2. Step-by-Step Transformation Trace & Preprocessing Ablation Study

Below we demonstrate the step-by-step transformation of a noisy civic report:

In [ ]:
sample_alert = (
    "<b>BREAKING:</b> Water supply won't resume in Ameerpet & Banjara Hills until 28/10/2026! "
    "Main pipeline burst near metro pillar #124. Tankers at https://ghmc.gov.in/tanker ⚠️"
)

trace = default_preprocessor.preprocess_with_steps(sample_alert)
print("Pipeline Configuration Trace:")
for s in trace["steps"]:
    print(f"  [{s['step']}]: {s['output']}")

In [ ]:
# Load and display saved Ablation Study results
ablation_file = PROJECT_ROOT / "reports" / "comparison_tables" / "preprocessing_ablation_study.csv"
if ablation_file.exists():
    df_ab = pd.read_csv(ablation_file)
    print("Preprocessing Ablation Study (Impact on Logistic Regression F1-Score):")
    display(df_ab)

## 3. Feature Extraction Comparison: BoW vs. TF-IDF vs. Word2Vec vs. FastText vs. BERT

We extract document representations across 5 distinct paradigms:
1. **Bag-of-Words (BoW):** Term frequency counts.
2. **TF-IDF:** Term frequency-inverse document frequency weighting unigrams and bigrams.
3. **Word2Vec (Continuous Bag-of-Words / Skip-Gram):** Dense semantic vectors trained on our civic corpus.
4. **FastText:** Dense vectors with character $n$-gram subwords (critical for rare or misspelled local place names).
5. **BERT / DistilBERT:** Deep bidirectional contextualized transformer embeddings.

In [ ]:
# Load seed reports
with open(PROJECT_ROOT / "seed" / "seed_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

non_spam = [d for d in data if not d.get("is_spam", False)]
texts = [d["text"] for d in non_spam]
categories = [d["category"] for d in non_spam]
cleaned = [default_preprocessor.preprocess(t) for t in texts]

# 1. BoW
bow_vec, X_bow = build_bow_vectorizer(cleaned, max_features=1000)

# 2. TF-IDF
tfidf_vec, X_tfidf = build_tfidf_vectorizer(cleaned, max_features=1000)

# 3. Word2Vec (Trained on corpus)
tokenized = [t.split() for t in cleaned]
w2v_model = train_word2vec(tokenized, vector_size=50)
X_w2v = np.array([get_word2vec_document_vector(w2v_model, tok, dim=50) for tok in tokenized])

# 4. FastText (Subwords for OOV names)
ft_model = train_fasttext(tokenized, vector_size=50)
X_ft = np.array([get_fasttext_document_vector(ft_model, tok, dim=50) for tok in tokenized])

print(f"BoW Shape: {X_bow.shape}")
print(f"TF-IDF Shape: {X_tfidf.shape}")
print(f"Word2Vec Shape: {X_w2v.shape}")
print(f"FastText Shape: {X_ft.shape}")

### FastText Advantage on Misspelled / OOV Indian Place Names
In civic reports, place names are frequently misspelled (e.g., `Kukatpally` typed as `Kukatpalli` or `Kukatpaly`).
- Word2Vec cannot generate a vector for words not seen in training.
- FastText builds representation from subword character $n$-grams, successfully producing a meaningful vector!

In [ ]:
target_word = "begumpet"
typo_word = "begumpett"

in_w2v = typo_word in w2v_model.wv
in_ft = typo_word in ft_model.wv  # True via subwords

print(f"Is '{typo_word}' in Word2Vec vocabulary? {in_w2v}")
print(f"Can FastText compute vector for '{typo_word}'? True")
print(f"FastText vector norm for '{typo_word}': {np.linalg.norm(ft_model.wv[typo_word]):.3f}")

## 4. Dimensionality Reduction & Manifold Visualization: PCA vs. t-SNE vs. UMAP

To visually understand how document embeddings cluster by civic category, we project our high-dimensional representations to 2D using:
- **PCA:** Linear projection maximizing global variance.
- **t-SNE:** Non-linear neighborhood preservation modeling local clustering.
- **UMAP / Spectral Manifold:** Non-linear topological projection preserving both local clusters and global domain relationships.

In [ ]:
# Display generated 3-panel visualization
from IPython.display import Image, display

img_path = PROJECT_ROOT / "reports" / "embedding_visualizations" / "pca_tsne_umap_comparison.png"
if img_path.exists():
    display(Image(filename=str(img_path)))

## 5. Written Interpretation of Embedding Manifolds
1. **PCA:** Captures the primary macro-split between atmospheric hazards (`severe_weather`) and surface infrastructure disruptions (`road_closure`, `water_supply`).
2. **t-SNE:** Effectively isolates high-specificity utility vocabularies (e.g. `power_cut` terms like 'transformer', 'megawatt', 'substation' form a tightly bounded satellite cluster).
3. **UMAP / Graph Manifold:** Preserves continuous transitions between categories where real-world causality exists (e.g., severe rainfall alerts transition smoothly into road closure clusters caused by flooding).